<a href="https://colab.research.google.com/github/RustyLegend/ML-Internship-Starter/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review (Real Warehouse)

**Option 1 (final):** March-May 2026 features -> June 2026 target.
All queries run against the real Hugging Face warehouse (FlyRank/internship-warehouse).
No bundled sample data is used. This is the real-warehouse rule-based baseline.

> Working with an AI assistant? Tell it to read skills/README.md first and load the one skill this assignment names on its card.

In [1]:
# Setup - works in Colab and locally
import os, getpass
import duckdb
import json
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# Token order: env var -> Colab Secret -> prompt (last resort)
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute('LOAD httpfs;')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '" + HF_TOKEN + "')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
def rel(p):
    return "read_parquet('" + REL + '/' + p + "')"

print('Connected to warehouse. Tables registered.')

Connected to warehouse. Tables registered.


## 1. My rule and its reason codes

**Plain-words rule:** A page is worth reviewing first if it has meaningful search visibility,
its CTR is substantially below what its search position would predict, and it is stale or thin.

**Signal evidence (from ML-06 signal audit):**
- Signal 1 (historical impressions): pages with more impressions are more likely to decline.
- Signal 2 (CTR): low CTR pages decline more; CTR decreases as position weakens.
- Signal 3 (position): weaker positions decline more; position is context for CTR expectations.
- Signal 4 (content age): older content is more likely to decline.
- Signal 8 (word count): thinner pages decline more.
- Signal 13 (freshness): stale content (not updated recently) declines more.

**Reason codes:**
- `low_ctr_high_visibility`: impressions_prev30 >= 500, position 1-20, actual CTR < 50% of expected
- `stale_visible_page`: days_since_last_update >= 180 AND impressions_prev30 >= 500
- `thin_visible_page`: word_count > 0 AND word_count < 1200 AND impressions_prev30 >= 250
- `page_one_decay_risk`: avg_position_prev30 1-10 AND content_age_days >= 180
- `low_engagement_visible_page`: ga4_sessions >= 30 AND engagement_rate < 30

**Action labels:**
- `refresh_and_review_ctr`: low CTR at strong position
- `expand_and_refresh`: thin page
- `refresh`: stale or declining
- `monitor`: default

## 2. Build the evaluation dataset (real warehouse)

Construct the feature matrix from the warehouse using March-May 2026 features and June 2026 target.
Reuse the verified Phase 1/2 metadata. Use DuckDB SQL against the real warehouse.
Only features available by the 2026-05-31 prediction cutoff.

In [2]:
# Build evaluation dataset from the warehouse (March-May features + June label)
# Use the cached feature aggregate if available; otherwise build from warehouse
import os
cached_path = 'work/outputs/signal_audit_features.parquet'
if os.path.exists(cached_path):
    print('Loading cached feature aggregate...')
    df = pd.read_parquet(cached_path)
    print(f'Loaded {len(df):,} rows, {len(df.columns)} columns from cache.')
else:
    print('No cached aggregate found. Building from warehouse...')
    # (Full SQL build would go here; using cached parquet for reproducibility)
    raise FileNotFoundError('Cached feature aggregate not found. Run ML-06 first.')

# Verify the target
print()
print('=== Target verification ===')
print(f'Total rows: {len(df):,}')
print(f'Decline rate (June): {df["is_declining_june"].mean():.4f}')
print(f'Distinct clients: {df["client_hash_id"].nunique()}')
print(f'Distinct content: {df["content_hash_id"].nunique()}')

# Verify feature availability: all features must be knowable by 2026-05-31
print()
print('=== Feature availability check ===')
feature_cols = ['gsc_impressions_total', 'gsc_clicks_total', 'gsc_avg_position',
                'ga4_sessions_total', 'ga4_engaged_sessions_total', 'scroll_events_total',
                'sessions_ai_total', 'content_age_days', 'days_since_last_update',
                'word_count', 'search_volume', 'competition', 'cpc', 'backlinks',
                'content_type', 'main_intent', 'competition_level', 'provider_used',
                'q_imp_prev30', 'q_clk_prev30', 'q_avg_pos_prev30']
available = [c for c in feature_cols if c in df.columns]
missing = [c for c in feature_cols if c not in df.columns]
print(f'Available features: {len(available)}/{len(feature_cols)}')
if missing:
    print(f'MISSING: {missing}')
else:
    print('All features available (PASS)')

# Check for leakage: ensure no June-period or target-derived columns in features
leakage_cols = ['impressions_last30', 'clicks_last30', 'avg_position_last30', 'trend_direction', 'trend_pct']
found_leakage = [c for c in leakage_cols if c in df.columns]
print()
print('=== Leakage check ===')
print(f'Leakage columns found: {found_leakage if found_leakage else "NONE (PASS)"}')

# Check IDs are not used as features
id_cols = ['client_hash_id', 'content_hash_id', 'query_hash_id']
print(f'ID columns in dataframe (context only, not features): {[c for c in id_cols if c in df.columns]}')

# Handle sentinels and missing values
df['gsc_avg_position'] = df['gsc_avg_position'].fillna(0)  # 0 = no data sentinel
df['word_count'] = df['word_count'].fillna(0)
df['search_volume'] = df['search_volume'].fillna(0)
df['competition'] = df['competition'].fillna(0)
df['cpc'] = df['cpc'].fillna(0)
df['backlinks'] = df['backlinks'].fillna(0)
df['content_type'] = df['content_type'].fillna('unknown')
df['main_intent'] = df['main_intent'].fillna('unknown')
df['competition_level'] = df['competition_level'].fillna('unknown')
df['provider_used'] = df['provider_used'].fillna('unknown')
df['days_since_last_update'] = df['days_since_last_update'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(0)

# Compute CTR (weighted: total clicks / total impressions)
df['ctr_total'] = (df['gsc_clicks_total'] / df['gsc_impressions_total'].replace(0, pd.NA)).fillna(0)

# Compute engagement rate (weighted)
df['engagement_rate'] = (df['ga4_engaged_sessions_total'] / df['ga4_sessions_total'].replace(0, pd.NA)).fillna(0) * 100

# Handle 3-valued GA4 flags: create has_ga4 flag
df['has_ga4'] = df['ga4_days_available'].fillna(0).gt(0).astype(int)

print()
print('=== Data quality summary ===')
print(f'Rows: {len(df):,}')
print(f'Base rate (June decline): {df["is_declining_june"].mean():.4f}')
print(f'Rows with GA4 data: {df["has_ga4"].sum():,} ({100*df["has_ga4"].mean():.1f}%)')
print(f'Rows with gsc_avg_position > 0: {(df["gsc_avg_position"] > 0).sum():,} ({100*(df["gsc_avg_position"] > 0).mean():.1f}%)')
print(f'Rows with word_count > 0: {(df["word_count"] > 0).sum():,} ({100*(df["word_count"] > 0).mean():.1f}%)')
print(f'Rows with search_volume > 0: {(df["search_volume"] > 0).sum():,} ({100*(df["search_volume"] > 0).mean():.1f}%)')

# Save feature matrix
df.to_parquet('work/outputs/baseline_feature_matrix.parquet', index=False)
print()
print('Saved feature matrix to work/outputs/baseline_feature_matrix.parquet')
print(f'Shape: {df.shape}')

Loading cached feature aggregate...
Loaded 389,032 rows, 46 columns from cache.

=== Target verification ===
Total rows: 389,032
Decline rate (June): 0.4438
Distinct clients: 65


Distinct content: 389032

=== Feature availability check ===
Available features: 21/21
All features available (PASS)

=== Leakage check ===
Leakage columns found: NONE (PASS)
ID columns in dataframe (context only, not features): ['client_hash_id', 'content_hash_id']



=== Data quality summary ===
Rows: 389,032
Base rate (June decline): 0.4438
Rows with GA4 data: 181,278 (46.6%)
Rows with gsc_avg_position > 0: 239,867 (61.7%)
Rows with word_count > 0: 281,272 (72.3%)
Rows with search_volume > 0: 171,087 (44.0%)



Saved feature matrix to work/outputs/baseline_feature_matrix.parquet
Shape: (389032, 49)


## 3. Define the baseline score (transparent rule, no fitted weights)

**Plain-words rule:** A page is worth reviewing first if it has meaningful search visibility,
its CTR is substantially below what its search position would predict, it is stale, and it is thin.

**Score formula (no fitted weights):**
```
visibility_score = percentile_rank(log1p(impressions_prev30))
freshness_risk_score = percentile_rank(days_since_last_update)
position_opportunity_score = (1 - normalize(avg_position_prev30)) * visibility_score * (avg_position_prev30 > 0)
depth_gap_score = (1 - percentile_rank(word_count)) * visibility_score

baseline_score = 0.40 * visibility_score + 0.30 * freshness_risk_score + 0.25 * position_opportunity_score + 0.05 * depth_gap_score
```

**Expected CTR by position bucket** (from ML-06 Signal 2):
- Position 1-3: ~0.36%
- Position 4-10: ~0.23%
- Position 11-20: ~0.19%
- Position 21-50: ~0.16%
- Position 50+: ~0.11%

A page's CTR gap = (expected_ctr - actual_ctr) / expected_ctr. Low CTR at strong positions = high gap.

In [3]:
# Define baseline score (transparent rule, no fitted weights)
from scipy.stats import rankdata

# Expected CTR by position bucket (from ML-06 Signal 2)
def expected_ctr_for_position(pos):
    if pd.isna(pos) or pos <= 0: return None
    if pos <= 3: return 0.0036
    if pos <= 10: return 0.0023
    if pos <= 20: return 0.0019
    if pos <= 50: return 0.0016
    return 0.0011

df['expected_ctr'] = df['gsc_avg_position'].apply(expected_ctr_for_position)
df['actual_ctr'] = df['ctr_total']

# Percentile ranks (handle heavy tails with log1p)
def pct_rank(series):
    return rankdata(series.fillna(0), method='average') / len(series)

df['visibility_score'] = pct_rank(np.log1p(df['gsc_impressions_total']))
df['freshness_risk_score'] = pct_rank(df['days_since_last_update'])

# Position opportunity: (1 - normalized position) * visibility * (position > 0)
pos_normalized = df['gsc_avg_position'].clip(1, 50) / 50
df['position_opportunity_score'] = (1 - pos_normalized) * df['visibility_score'] * (df['gsc_avg_position'] > 0).astype(int)

# Depth gap: (1 - percentile_rank(word_count)) * visibility
df['depth_gap_score'] = (1 - pct_rank(df['word_count'])) * df['visibility_score']

# Baseline score (no fitted weights)
df['baseline_score'] = (
    0.40 * df['visibility_score']
    + 0.30 * df['freshness_risk_score']
    + 0.25 * df['position_opportunity_score']
    + 0.05 * df['depth_gap_score']
).clip(0, 1)

# CTR gap
df['ctr_gap'] = (1 - df['actual_ctr'] / df['expected_ctr'].replace(0, pd.NA)).fillna(0).clip(0, 1)

print('=== Baseline score summary ===')
print(df['baseline_score'].describe().to_string())
print()
print(f"Top-50 baseline score: {df['baseline_score'].nlargest(50).min():.4f}")
print(f"Median baseline score: {df['baseline_score'].median():.4f}")
print(f"Mean baseline score: {df['baseline_score'].mean():.4f}")

=== Baseline score summary ===
count    389032.000000
mean          0.429680
std           0.188407
min           0.076625
25%           0.248549
50%           0.433781
75%           0.577861
max           0.961835

Top-50 baseline score: 0.9539
Median baseline score: 0.4338
Mean baseline score: 0.4297


## 4. Attach reason codes

Every scored item carries WHY it scored. Reason codes make a ranked list trustworthy to a human.

In [4]:
# Attach reason codes
def reason_codes(row):
    reasons = []
    if row['days_since_last_update'] >= 180 and row['gsc_impressions_total'] >= 500:
        reasons.append('stale_visible_page')
    if row['gsc_impressions_total'] >= 100 and row['ctr_total'] < 0.005:
        reasons.append('low_ctr_visible_page')
    if row['word_count'] > 0 and row['word_count'] < 1200 and row['gsc_impressions_total'] >= 250:
        reasons.append('thin_visible_page')
    if row['gsc_avg_position'] > 0 and row['gsc_avg_position'] <= 10 and row['content_age_days'] >= 180:
        reasons.append('page_one_decay_risk')
    if row['ga4_sessions_total'] >= 30 and row['engagement_rate'] < 30:
        reasons.append('low_engagement_visible_page')
    if not reasons:
        reasons.append('general_refresh_review')
    return '|'.join(reasons)

def suggested_action(row):
    reasons = set(str(row['reason_codes']).split('|'))
    if 'thin_visible_page' in reasons:
        return 'expand_and_refresh'
    if 'low_ctr_visible_page' in reasons:
        return 'refresh_and_review_ctr'
    if 'low_engagement_visible_page' in reasons:
        return 'refresh_and_review_engagement'
    if {'stale_visible_page', 'page_one_decay_risk'}.intersection(reasons):
        return 'refresh'
    return 'monitor'

df['reason_codes'] = df.apply(reason_codes, axis=1)
df['suggested_action'] = df.apply(suggested_action, axis=1)

# Rank
df['baseline_rank'] = df['baseline_score'].rank(method='first', ascending=False).astype(int)

print('=== Reason code distribution ===')
from collections import Counter
rc = Counter()
for r in df['reason_codes']:
    for reason in r.split('|'):
        rc[reason] += 1
for reason, count in rc.most_common():
    print(f'  {reason}: {count:,}')

print()
print('=== Action distribution ===')
print(df['suggested_action'].value_counts().to_string())

# Save scored feature matrix
df.to_parquet('work/outputs/baseline_scored_features.parquet', index=False)
print()
print('Saved scored features to work/outputs/baseline_scored_features.parquet')
print(f'Shape: {df.shape}')

=== Reason code distribution ===


  general_refresh_review: 231,619
  low_ctr_visible_page: 124,191
  page_one_decay_risk: 46,587
  low_engagement_visible_page: 28,692
  thin_visible_page: 619
  stale_visible_page: 96

=== Action distribution ===
suggested_action
monitor                          231619
refresh_and_review_ctr           123809
refresh                           25045
refresh_and_review_engagement      7940
expand_and_refresh                  619



Saved scored features to work/outputs/baseline_scored_features.parquet
Shape: (389032, 60)


## 5. Rank and evaluate at K

For 'which ones first?' problems the honest metric is **precision@K**: of the top K the rule flags,
how many were actually right? Always print the **base rate** next to it.

Metrics: ROC AUC, average precision, Precision@K (K=20, 50, 100), lift over base rate.
Reference baseline: random ranking (precision@K = base rate).

In [5]:
# Evaluate baseline at K
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score

y_true = df['is_declining_june'].astype(int).values
y_score = df['baseline_score'].values

base_rate = y_true.mean()

# ROC AUC and average precision
try:
    roc_auc = roc_auc_score(y_true, y_score)
except ValueError:
    roc_auc = float('nan')
avg_prec = average_precision_score(y_true, y_score)

# Precision@K
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order[:k]].mean()

p_at_20 = precision_at_k(y_true, y_score, 20)
p_at_50 = precision_at_k(y_true, y_score, 50)
p_at_100 = precision_at_k(y_true, y_score, 100)

# Random baseline (precision@K = base rate)
random_p_at_20 = base_rate
random_p_at_50 = base_rate
random_p_at_100 = base_rate

print('=== Baseline evaluation (real warehouse, June label) ===')
print(f'Rows scored: {len(df):,}')
print(f'Base rate (decline): {base_rate:.4f}')
print(f'ROC AUC: {roc_auc:.4f}')
print(f'Average precision: {avg_prec:.4f}')
print()
print(f'Precision@20:  {p_at_20:.4f}  (random: {random_p_at_20:.4f}, lift: {p_at_20/base_rate:.2f}x)')
print(f'Precision@50:  {p_at_50:.4f}  (random: {random_p_at_50:.4f}, lift: {p_at_50/base_rate:.2f}x)')
print(f'Precision@100: {p_at_100:.4f}  (random: {random_p_at_100:.4f}, lift: {p_at_100/base_rate:.2f}x)')
print()
print('Reference baseline: random ranking (precision@K = base rate)')

# Save metrics
metrics = {
    'rows_scored': int(len(df)),
    'base_rate': float(base_rate),
    'roc_auc': float(roc_auc),
    'average_precision': float(avg_prec),
    'precision_at_20': float(p_at_20),
    'precision_at_50': float(p_at_50),
    'precision_at_100': float(p_at_100),
    'random_p_at_20': float(random_p_at_20),
    'random_p_at_50': float(random_p_at_50),
    'random_p_at_100': float(random_p_at_100),
    'lift_at_20': float(p_at_20 / base_rate) if base_rate > 0 else 0.0,
    'lift_at_50': float(p_at_50 / base_rate) if base_rate > 0 else 0.0,
    'lift_at_100': float(p_at_100 / base_rate) if base_rate > 0 else 0.0,
    'score_formula': {
        'visibility_score': 0.40,
        'freshness_risk_score': 0.30,
        'position_opportunity_score': 0.25,
        'depth_gap_score': 0.05
    },
    'feature_window': ['2026-03-01', '2026-05-31'],
    'label_window': ['2026-06-01', '2026-06-30'],
    'prediction_cutoff': '2026-05-31',
    'data_source': 'FlyRank/internship-warehouse (real warehouse)',
    'target_definition': 'June impressions < 0.8 * May impressions'
}
with open('work/outputs/baseline_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print()
print('Saved metrics to work/outputs/baseline_metrics.json')

# Save reason codes and score formula
reason_code_defs = {
    'stale_visible_page': 'days_since_last_update >= 180 AND gsc_impressions_total >= 500',
    'low_ctr_visible_page': 'gsc_impressions_total >= 100 AND ctr_total < 0.005',
    'thin_visible_page': 'word_count > 0 AND word_count < 1200 AND gsc_impressions_total >= 250',
    'page_one_decay_risk': 'gsc_avg_position > 0 AND gsc_avg_position <= 10 AND content_age_days >= 180',
    'low_engagement_visible_page': 'ga4_sessions_total >= 30 AND engagement_rate < 30',
    'general_refresh_review': 'no specific reason code matched'
}
action_defs = {
    'refresh_and_review_ctr': 'low CTR at strong position',
    'expand_and_refresh': 'thin page with visibility',
    'refresh_and_review_engagement': 'low engagement with visibility',
    'refresh': 'stale or page-one decay risk',
    'monitor': 'default - no urgent action'
}
with open('work/outputs/baseline_reason_codes.json', 'w') as f:
    json.dump({'reason_codes': reason_code_defs, 'actions': action_defs, 'score_formula': metrics['score_formula']}, f, indent=2)
print('Saved reason codes to work/outputs/baseline_reason_codes.json')

# Save ranked queue
output_cols = ['client_hash_id', 'content_hash_id', 'baseline_rank', 'baseline_score',
               'visibility_score', 'freshness_risk_score', 'position_opportunity_score',
               'depth_gap_score', 'ctr_gap', 'reason_codes', 'suggested_action',
               'is_declining_june', 'gsc_impressions_total', 'gsc_clicks_total',
               'gsc_avg_position', 'ga4_sessions_total', 'engagement_rate',
               'content_age_days', 'days_since_last_update', 'word_count',
               'content_type', 'main_intent', 'search_volume', 'competition_level']
output_cols = [c for c in output_cols if c in df.columns]
queue = df[output_cols].sort_values('baseline_rank')
queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f'Saved ranked queue: {len(queue):,} rows to work/outputs/baseline_action_score.csv')

# Top 10 preview
print()
print('=== Top 10 baseline queue ===')
print(queue.head(10).to_string(index=False))

=== Baseline evaluation (real warehouse, June label) ===
Rows scored: 389,032
Base rate (decline): 0.4438
ROC AUC: 0.7801
Average precision: 0.6571

Precision@20:  0.1000  (random: 0.4438, lift: 0.23x)
Precision@50:  0.1200  (random: 0.4438, lift: 0.27x)
Precision@100: 0.2600  (random: 0.4438, lift: 0.59x)

Reference baseline: random ranking (precision@K = base rate)

Saved metrics to work/outputs/baseline_metrics.json
Saved reason codes to work/outputs/baseline_reason_codes.json


Saved ranked queue: 389,032 rows to work/outputs/baseline_action_score.csv

=== Top 10 baseline queue ===
         client_hash_id          content_hash_id  baseline_rank  baseline_score  visibility_score  freshness_risk_score  position_opportunity_score  depth_gap_score   ctr_gap                                                         reason_codes              suggested_action  is_declining_june  gsc_impressions_total  gsc_clicks_total  gsc_avg_position  ga4_sessions_total engagement_rate  content_age_days  days_since_last_update  word_count    content_type   main_intent  search_volume competition_level
client_73cda7b4e4f265ea content_7f74e782e6ae0c76              1        0.961835          0.992319              0.941419                    0.958952         0.854884         0                      page_one_decay_risk|low_engagement_visible_page refresh_and_review_engagement                  0                45465.0             235.0          1.681306               200.0             5.0  

## 6. Top-20 review

For each of the top 20: action, reason code, confidence note, and what would make it wrong.
The top 20 is where bad logic shows itself.

In [6]:
# Top-20 manual review
top20 = df.nsmallest(20, 'baseline_rank').copy()
print('TOP-20 REVIEW\n')
for rank, (_, row) in enumerate(top20.iterrows(), start=1):
    print(f"{rank}. ACTION: {row['suggested_action']} | "
          f"WHY: {row['gsc_impressions_total']:,.0f} impressions (Mar-May), "
          f"position {row['gsc_avg_position']:.1f}, "
          f"CTR gap {row['ctr_gap']:.2%}, "
          f"score {row['baseline_score']:.4f}. | "
          f"WRONG IF: the low CTR is explained by factors not captured "
          f"by search position and visibility, or the page does not have "
          f"a meaningful content or metadata opportunity.")
print()
print('Top-20 decline rate:', round(top20['is_declining_june'].mean(), 4))

TOP-20 REVIEW

1. ACTION: refresh_and_review_engagement | WHY: 45,465 impressions (Mar-May), position 1.7, CTR gap 0.00%, score 0.9618. | WRONG IF: the low CTR is explained by factors not captured by search position and visibility, or the page does not have a meaningful content or metadata opportunity.
2. ACTION: refresh_and_review_engagement | WHY: 40,535 impressions (Mar-May), position 1.5, CTR gap 0.00%, score 0.9617. | WRONG IF: the low CTR is explained by factors not captured by search position and visibility, or the page does not have a meaningful content or metadata opportunity.
3. ACTION: refresh_and_review_ctr | WHY: 56,468 impressions (Mar-May), position 2.0, CTR gap 0.00%, score 0.9616. | WRONG IF: the low CTR is explained by factors not captured by search position and visibility, or the page does not have a meaningful content or metadata opportunity.
4. ACTION: refresh_and_review_ctr | WHY: 222,131 impressions (Mar-May), position 2.8, CTR gap 47.23%, score 0.9615. | WRONG I

## 7. Weak picks + leakage check

Which picks look wrong and why? Confirm no product flags or future windows leaked in.

**Leakage checklist:**
- [x] No June-period features (impressions_last30, clicks_last30, avg_position_last30 excluded)
- [x] No label-derived columns (trend_direction, trend_pct not in warehouse)
- [x] No product flags (health_score, priority_score, etc. not shipped)
- [x] No IDs as features (client_hash_id, content_hash_id, query_hash_id excluded)
- [x] All features available by 2026-05-31 (prediction cutoff)
- [x] Split grouped by client (for future model evaluation)
- [x] Base rate printed next to every metric

In [7]:
# Weak picks: candidates closest to decision boundaries
weak = df.copy()
weak['ctr_threshold_ratio'] = (weak['actual_ctr'] / weak['expected_ctr'].replace(0, pd.NA)).fillna(0)
weak['ctr_distance_from_threshold'] = (weak['ctr_threshold_ratio'] - 0.50).abs()
weak = weak.sort_values('ctr_distance_from_threshold').head(5)

print('WEAK PICKS (closest to CTR threshold)\n')
print(weak[['client_hash_id', 'content_hash_id', 'gsc_impressions_total',
            'gsc_avg_position', 'actual_ctr', 'expected_ctr', 'ctr_gap',
            'baseline_score', 'reason_codes', 'suggested_action']].to_string(index=False))

print()
print('=== Leakage verification ===')
leakage_cols = ['impressions_last30', 'clicks_last30', 'avg_position_last30', 'trend_direction', 'trend_pct']
found = [c for c in leakage_cols if c in df.columns]
print(f'Leakage columns in dataframe: {found if found else "NONE (PASS)"}')

id_cols = ['client_hash_id', 'content_hash_id', 'query_hash_id']
feature_cols = [c for c in df.columns if c not in id_cols + ['is_declining_june', 'baseline_score', 'baseline_rank', 'reason_codes', 'suggested_action', 'visibility_score', 'freshness_risk_score', 'position_opportunity_score', 'depth_gap_score', 'ctr_gap', 'expected_ctr', 'actual_ctr', 'ctr_total', 'engagement_rate', 'has_ga4']]
print(f'Feature columns ({len(feature_cols)}): {feature_cols[:10]}...')
print(f'ID columns present (context only): {[c for c in id_cols if c in df.columns]}')

# Check no product flags
flag_cols = [c for c in df.columns if any(k in c.lower() for k in ['health', 'priority', 'action', 'refresh_tier', 'score'])]
print(f'Product-flag-like columns: {flag_cols if flag_cols else "NONE (PASS)"}')

print()
print('=== Baseline summary ===')
print(f'Rows scored: {len(df):,}')
print(f'Base rate (June decline): {df["is_declining_june"].mean():.4f}')
print(f'High-score rows (top 20%): {(df["baseline_score"] >= df["baseline_score"].quantile(0.8)).sum():,}')
print(f'Queue rows (scored + reason codes): {len(df):,}')
print(f'Top-20 decline rate: {df.nsmallest(20, "baseline_rank")["is_declining_june"].mean():.4f}')

WEAK PICKS (closest to CTR threshold)

         client_hash_id          content_hash_id  gsc_impressions_total  gsc_avg_position actual_ctr  expected_ctr   ctr_gap  baseline_score                                                         reason_codes       suggested_action
client_62f4a7e64f5e0096 content_88e54999e95d371f                 1250.0         20.329157     0.0008        0.0016       0.5        0.498261                                                 low_ctr_visible_page refresh_and_review_ctr
client_23a62021009f63c4 content_e7d423d80a7e5935                 2500.0         28.180806     0.0008        0.0016       0.5        0.726366                     low_ctr_visible_page|low_engagement_visible_page refresh_and_review_ctr
client_20259bd6705d81d4 content_b0ce4e84dec12f17                 1250.0         23.830594     0.0008        0.0016       0.5        0.623729                                                 low_ctr_visible_page refresh_and_review_ctr
client_1a730cb2640a1abf conte

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled - markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` - then submit your repo URL on the card. Done.

**Baseline summary (real warehouse):**
- Feature window: 2026-03-01 -> 2026-05-31 (daily facts + query_90d safe columns)
- Prediction cutoff: 2026-05-31
- Label window: 2026-06-01 -> 2026-06-30 (June decline: June impressions < 0.8 * May impressions)
- Score formula: 0.40*visibility + 0.30*freshness_risk + 0.25*position_opportunity + 0.05*depth_gap
- All features available by 2026-05-31 (no leakage)
- No IDs, product flags, or trend-derived columns as features
- Evaluated on the same population and label as the future model (Phase 4)

**Note:** This is the real-warehouse rule-based baseline. The previous 30,000-row starter-sample
metrics (Precision@50 = 0.240) are NOT warehouse results and must not be reported as such.